In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder, StandardScaler
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array, array_to_vector

# Models:
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.classification import MultilayerPerceptronClassifier

In [0]:
auc_eval = BinaryClassificationEvaluator(
    labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
recall_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="recallByLabel"
)
f1_eval = MulticlassClassificationEvaluator(
    labelCol="label", predictionCol="prediction", metricName="fMeasureByLabel"
)

In [0]:
train=spark.read.table("teams.sensorx.train_data")
test=spark.read.table("teams.sensorx.test_data")

n_lags = 20
H_days = 7

## Gradient Boosted Trees Model Training and Evaluation

In [0]:
# Class weighting (give minority class higher weight)
label_counts = train.groupBy("label").count().collect()
total = sum(row["count"] for row in label_counts)
weight_map = {row["label"]: total / (2.0 * row["count"]) for row in label_counts}

for lbl in sorted(weight_map):
    cnt = [r["count"] for r in label_counts if r["label"] == lbl][0]
    pct = cnt / total * 100
    print(f"  Label {lbl}: {cnt:,} samples ({pct:.1f}%) -> weight = {weight_map[lbl]:.4f}")
print(f"  Total: {total:,} samples")

# Add weight column to train and test
weight_expr = F.when(F.col("label") == 1, weight_map[1]).otherwise(weight_map[0])
train_w = train.withColumn("weight", weight_expr)
test_w = test.withColumn("weight", weight_expr)

In [0]:
# Strip corrupted ML metadata from features vector (fixes issues with Spark ML models after saving/loading)
train_gbt = train_w.withColumn("features", array_to_vector(vector_to_array("features")))
test_gbt = test_w.withColumn("features", array_to_vector(vector_to_array("features")))

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=30,
    maxBins=64,
    seed=42
)

GBTmodel = gbt.fit(train_gbt)
gbtpredictions = GBTmodel.transform(test_gbt)

# --- Training accuracy ---
gbt_train_preds = GBTmodel.transform(train_gbt)
GBT_train_auc = auc_eval.evaluate(gbt_train_preds)

# --- Test evaluation ---
GBT_auc = auc_eval.evaluate(gbtpredictions)

# Recall (per class)
GBT_recall_0 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 0.0})
GBT_recall_1 = recall_eval.evaluate(gbtpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
GBT_f1_0 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 0.0})
GBT_f1_1 = f1_eval.evaluate(gbtpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nGBT with {n_lags} lags + deviceId (class-weighted)")
print(f"Training AUC: {GBT_train_auc:.4f}")
print(f"Test AUC:     {GBT_auc:.4f}")
print(f"Recall (label=0): {GBT_recall_0:.4f}")
print(f"Recall (label=1): {GBT_recall_1:.4f}")
print(f"F1     (label=0): {GBT_f1_0:.4f}")
print(f"F1     (label=1): {GBT_f1_1:.4f}")

# Confusion matrix
gbtpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
# --- Feature importance ---
# Fetch feature names from the GBT model's input column metadata
input_col = GBTmodel.getFeaturesCol()
feature_metadata = train_gbt.schema[input_col].metadata
if "ml_attr" in feature_metadata and "attrs" in feature_metadata["ml_attr"]:
    attrs = feature_metadata["ml_attr"]["attrs"]
    feature_names = []
    for attr_type in ["numeric", "binary"]:
        if attr_type in attrs:
            feature_names.extend([attr["name"] for attr in attrs[attr_type]])
else:
    num_features = GBTmodel.numFeatures
    feature_names = [f"feature_{i}" for i in range(num_features)]

importances = GBTmodel.featureImportances.toArray().tolist()
feat_imp_df = spark.createDataFrame(
    list(zip(feature_names, importances)), ["feature", "importance"]
).orderBy(F.desc("importance"))

print(f"\nTotal features: {GBTmodel.numFeatures}")
print(f"Top 20 features by importance:")
display(feat_imp_df.limit(20))

In [0]:
# --- Time-Based Gradient Boosted Trees ---
# Extract temporal features from timeStamp and append to existing feature vector
from pyspark.ml.functions import vector_to_array, array_to_vector

# Extract time features
def add_time_features(df):
    return (
        df
        .withColumn("hour", F.hour("timeStamp"))
        .withColumn("dayOfWeek", F.dayofweek("timeStamp"))  # 1=Sun, 7=Sat
        .withColumn("dayOfMonth", F.dayofmonth("timeStamp"))
        .withColumn("month", F.month("timeStamp"))
    )

train_time = add_time_features(train_w)
test_time = add_time_features(test_w)

# Convert existing features vector to array, append time features, reassemble
time_feature_cols = ["hour", "dayOfWeek", "dayOfMonth", "month"]

train_time = (
    train_time
    .withColumn("features_arr", vector_to_array("features"))
    .withColumn(
        "features_combined",
        array_to_vector(
            F.concat(
                F.col("features_arr"),
                F.array([F.col(c).cast("double") for c in time_feature_cols])
            )
        )
    )
    .drop("features", "features_arr")
    .withColumnRenamed("features_combined", "features")
)

test_time = (
    test_time
    .withColumn("features_arr", vector_to_array("features"))
    .withColumn(
        "features_combined",
        array_to_vector(
            F.concat(
                F.col("features_arr"),
                F.array([F.col(c).cast("double") for c in time_feature_cols])
            )
        )
    )
    .drop("features", "features_arr")
    .withColumnRenamed("features_combined", "features")
)

# Train time-based GBT
gbt_time = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=30,
    maxBins=64,
    seed=42
)

GBT_time_model = gbt_time.fit(train_time)
gbt_time_preds = GBT_time_model.transform(test_time)

# --- Training accuracy ---
gbt_time_train_preds = GBT_time_model.transform(train_time)
GBT_time_train_auc = auc_eval.evaluate(gbt_time_train_preds)

# --- Test evaluation ---
GBT_time_auc = auc_eval.evaluate(gbt_time_preds)

# Recall (per class)
GBT_time_recall_0 = recall_eval.evaluate(gbt_time_preds, {recall_eval.metricLabel: 0.0})
GBT_time_recall_1 = recall_eval.evaluate(gbt_time_preds, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
GBT_time_f1_0 = f1_eval.evaluate(gbt_time_preds, {f1_eval.metricLabel: 0.0})
GBT_time_f1_1 = f1_eval.evaluate(gbt_time_preds, {f1_eval.metricLabel: 1.0})

print(f"\nTime-Based GBT with {n_lags} lags + deviceId + time features (class-weighted)")
print(f"Added features: {time_feature_cols}")
print(f"Total features: {GBT_time_model.numFeatures} (original + {len(time_feature_cols)} time features)")
print(f"Training AUC: {GBT_time_train_auc:.4f}")
print(f"Test AUC:     {GBT_time_auc:.4f}")
print(f"Recall (label=0): {GBT_time_recall_0:.4f}")
print(f"Recall (label=1): {GBT_time_recall_1:.4f}")
print(f"F1     (label=0): {GBT_time_f1_0:.4f}")
print(f"F1     (label=1): {GBT_time_f1_1:.4f}")

# Confusion matrix
print("\nConfusion matrix:")
gbt_time_preds.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
# --- Downsampled Time-Based GBT ---
# Strategy: Downsample majority class (label=0) to ~3:1 ratio instead of class weighting.
# This reduces false positives by giving the model fewer "normal" examples to memorize.
# Combined with: time features, maxDepth=7, maxIter=50

# --- Step 1: Downsample label 0 ---
# Current ratio is ~7:1 (87% vs 13%). Target: 3:1
train_label_1 = train.filter(F.col("label") == 1)
train_label_0 = train.filter(F.col("label") == 0)

count_1 = train_label_1.count()
count_0 = train_label_0.count()
target_ratio = 3.0  # 3 negatives per 1 positive
fraction_to_keep = (target_ratio * count_1) / count_0

train_label_0_sampled = train_label_0.sample(withReplacement=False, fraction=fraction_to_keep, seed=42)
train_downsampled = train_label_0_sampled.unionByName(train_label_1)

new_count_0 = train_label_0_sampled.count()
print(f"Original: {count_0:,} label=0, {count_1:,} label=1 (ratio {count_0/count_1:.1f}:1)")
print(f"Downsampled: {new_count_0:,} label=0, {count_1:,} label=1 (ratio {new_count_0/count_1:.1f}:1)")
print(f"Training set reduced from {count_0+count_1:,} to {new_count_0+count_1:,} samples")

# --- Step 2: Add time features and assemble ---
train_ds = add_time_features(train_downsampled)
test_ds = add_time_features(test)  # Do NOT downsample test set

for name, df in [("train", train_ds), ("test", test_ds)]:
    df_out = (
        df
        .withColumn("features_arr", vector_to_array("features"))
        .withColumn(
            "features_combined",
            array_to_vector(
                F.concat(
                    F.col("features_arr"),
                    F.array([F.col(c).cast("double") for c in time_feature_cols])
                )
            )
        )
        .drop("features", "features_arr")
        .withColumnRenamed("features_combined", "features")
    )
    if name == "train":
        train_ds = df_out
    else:
        test_ds = df_out

# --- Step 3: Train GBT (no class weight needed with downsampling) ---
gbt_ds = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    maxDepth=7,
    maxBins=64,
    seed=42
)

GBT_ds_model = gbt_ds.fit(train_ds)
gbt_ds_preds = GBT_ds_model.transform(test_ds)

# --- Default threshold metrics ---
GBT_ds_train_auc = auc_eval.evaluate(GBT_ds_model.transform(train_ds))
GBT_ds_auc = auc_eval.evaluate(gbt_ds_preds)
GBT_ds_f1_1 = f1_eval.evaluate(gbt_ds_preds, {f1_eval.metricLabel: 1.0})
GBT_ds_recall_1 = recall_eval.evaluate(gbt_ds_preds, {recall_eval.metricLabel: 1.0})
GBT_ds_recall_0 = recall_eval.evaluate(gbt_ds_preds, {recall_eval.metricLabel: 0.0})
GBT_ds_f1_0 = f1_eval.evaluate(gbt_ds_preds, {f1_eval.metricLabel: 0.0})

print(f"\nDownsampled Time-GBT (3:1 ratio, maxDepth=7, maxIter=50, NO class weighting)")
print(f"Training AUC: {GBT_ds_train_auc:.4f}")
print(f"Test AUC:     {GBT_ds_auc:.4f}")
print(f"Recall (label=0): {GBT_ds_recall_0:.4f}  |  F1 (label=0): {GBT_ds_f1_0:.4f}")
print(f"Recall (label=1): {GBT_ds_recall_1:.4f}  |  F1 (label=1): {GBT_ds_f1_1:.4f}")

# Confusion matrix
print("\nConfusion matrix (default threshold=0.5):")
gbt_ds_preds.groupBy("label").pivot("prediction").count().orderBy("label").show()

# --- Step 4: Threshold tuning ---
gbt_ds_probs = gbt_ds_preds.withColumn("prob_1", vector_to_array("probability")[1])

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
results_ds = []

print("Threshold tuning:")
for thresh in thresholds:
    df_t = gbt_ds_probs.withColumn(
        "prediction", F.when(F.col("prob_1") >= thresh, 1.0).otherwise(0.0)
    )
    tp = df_t.filter((F.col("label") == 1) & (F.col("prediction") == 1.0)).count()
    fp = df_t.filter((F.col("label") == 0) & (F.col("prediction") == 1.0)).count()
    fn = df_t.filter((F.col("label") == 1) & (F.col("prediction") == 0.0)).count()
    tn = df_t.filter((F.col("label") == 0) & (F.col("prediction") == 0.0)).count()
    
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    
    results_ds.append((thresh, prec, rec, f1, tp, fp, fn))
    print(f"  {thresh:.2f} | Prec={prec:.4f} | Recall={rec:.4f} | F1={f1:.4f} | TP={tp:,} FP={fp:,}")

best_ds = max(results_ds, key=lambda x: x[3])
print(f"\n>>> BEST F1(label=1): {best_ds[3]:.4f} at threshold {best_ds[0]:.2f}")
print(f"    Precision={best_ds[1]:.4f}, Recall={best_ds[2]:.4f}")
print(f"\n--- Comparison ---")
print(f"    Original Time-GBT (weighted, thresh=0.50):     F1=0.4967")
print(f"    Time-GBT (weighted, best thresh=0.75):         F1=0.5531")
print(f"    Downsampled Time-GBT (best thresh={best_ds[0]:.2f}):    F1={best_ds[3]:.4f}")

## Random Forest Model Training and Evaluation


In [0]:
# --- RandomForest with class weighting ---
# Strip corrupted ML metadata from features vector (same fix as GBT)
from pyspark.ml.functions import vector_to_array, array_to_vector

n_lags = 20
train_rf = train_w.withColumn("features", array_to_vector(vector_to_array("features")))
test_rf = test_w.withColumn("features", array_to_vector(vector_to_array("features")))

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=100,
    maxBins=64,
    seed=42
)
RFmodel = rf.fit(train_rf)
RFpredictions = RFmodel.transform(test_rf)

RF_train_auc = auc_eval.evaluate(RFmodel.transform(train_rf))
RF_auc = auc_eval.evaluate(RFpredictions)

# Recall (per class)
RF_recall_0 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 0.0})
RF_recall_1 = recall_eval.evaluate(RFpredictions, {recall_eval.metricLabel: 1.0})

# F1 score (per class)
RF_f1_0 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 0.0})
RF_f1_1 = f1_eval.evaluate(RFpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nRandom Forest with {n_lags} lags + deviceId (class-weighted)")
print(f"Training AUC: {RF_train_auc:.4f}")
print(f"Test AUC:     {RF_auc:.4f}")
print(f"Recall (label=0): {RF_recall_0:.4f}")
print(f"Recall (label=1): {RF_recall_1:.4f}")
print(f"F1     (label=0): {RF_f1_0:.4f}")
print(f"F1     (label=1): {RF_f1_1:.4f}")

# Confusion matrix
RFpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
# --- Feature importance ---
ohe_size = RFmodel.numFeatures - len(sensor_cols) - len(lag_cols)
ohe_names = [f"deviceId_ohe_{i}" for i in range(ohe_size)]
all_feature_names = sensor_cols + lag_cols + ohe_names

rf_importances = RFmodel.featureImportances.toArray().tolist()
rf_feat_imp_df = spark.createDataFrame(
    zip(all_feature_names, rf_importances), ["feature", "importance"]
).orderBy(F.desc("importance"))

print(f"\nTotal features: {RFmodel.numFeatures} ({len(sensor_cols)} sensor + {len(lag_cols)} lag + {ohe_size} OHE)")
print(f"Top 20 features by importance:")
display(rf_feat_imp_df.limit(20))

## Multilayer Perceptron Classifier Model Training and Evaluation

In [0]:
# --- Normalize features for MLP ---
# Fit scaler on full train data
scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withMean=True,
    withStd=True
)
scaler_model = scaler.fit(train)

# Transform train and test
train_norm = scaler_model.transform(train).drop("features").withColumnRenamed("features_scaled", "features")
test_norm = scaler_model.transform(test).drop("features").withColumnRenamed("features_scaled", "features")

print(f"Training set size: {train_norm.count():,}")
print(f"Test set size: {test_norm.count():,}")
print(f"Normalized feature vector size: {train_norm.select('features').first()[0].size}")

In [0]:
# Input layer = 133 features, output = 2 classes (binary horizon)
layers = [133, 16, 16, 2] 
H_days = 7
n_lags = 20

trainer = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    layers=layers,
    blockSize=128,
    seed=42,
    stepSize=0.001
)

MLPmodel = trainer.fit(train_norm)
MLPpredictions = MLPmodel.transform(test_norm)

# --- Evaluate ---
MLP_training_auc = auc_eval.evaluate(MLPmodel.transform(train_norm))
MLP_auc = auc_eval.evaluate(MLPpredictions)

MLP_recall_0 = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: 0.0})
MLP_recall_1 = recall_eval.evaluate(MLPpredictions, {recall_eval.metricLabel: 1.0})
MLP_f1_0 = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: 0.0})
MLP_f1_1 = f1_eval.evaluate(MLPpredictions, {f1_eval.metricLabel: 1.0})

print(f"\nMLP Neural Network (binary {H_days}-day horizon, {n_lags} lags + deviation features)")
print(f"Layers: {layers}")
print(f"Training AUC: {MLP_training_auc:.4f}")
print(f"Test AUC:     {MLP_auc:.4f}")
print(f"  Class 0 (safe):  Recall={MLP_recall_0:.4f}  F1={MLP_f1_0:.4f}")
print(f"  Class 1 (fault): Recall={MLP_recall_1:.4f}  F1={MLP_f1_1:.4f}")

# Confusion matrix
print("\nConfusion matrix:")
MLPpredictions.groupBy("label").pivot("prediction").count().orderBy("label").show()

In [0]:
import os
import mlflow
from mlflow.models import infer_signature
from pyspark.ml.functions import vector_to_array

# Required for Spark ML model logging on shared/serverless clusters
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/teams/sensorx/data-dump/mlflow_tmp"

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment("/Shared/sensorx_mlp_fault_prediction")

CATALOG = "teams"
SCHEMA = "sensorx"
MLP_MODEL_NAME = f"{CATALOG}.{SCHEMA}.mlp_fault_prediction"

with mlflow.start_run(run_name="MLP_fault_prediction_v4") as run:
    # Log parameters
    mlflow.log_param("model_type", "MultilayerPerceptronClassifier")
    mlflow.log_param("layers", str(layers))
    mlflow.log_param("maxIter", 100)
    mlflow.log_param("stepSize", 0.001)
    mlflow.log_param("blockSize", 128)
    mlflow.log_param("n_lags", n_lags)
    mlflow.log_param("H_days", H_days)
    mlflow.log_param("normalized", True)
    mlflow.log_param("deviation_features", True)
    mlflow.log_param("downsampled", False)

    # Log metrics
    mlflow.log_metric("AUC", MLP_auc)
    mlflow.log_metric("Training_AUC", MLP_training_auc)
    mlflow.log_metric("Recall_label_0", MLP_recall_0)
    mlflow.log_metric("Recall_label_1", MLP_recall_1)
    mlflow.log_metric("F1_label_0", MLP_f1_0)
    mlflow.log_metric("F1_label_1", MLP_f1_1)

    # Infer signature from a small sample
    sample_input = test_norm.select("features").limit(5)
    sample_output = MLPmodel.transform(sample_input).select("prediction")
    signature = infer_signature(
        sample_input.toPandas(),
        sample_output.toPandas()
    )

    # Convert DenseVector to array for JSON-serializable input_example
    input_example = (
        sample_input.limit(1)
        .select(vector_to_array("features").alias("features"))
        .toPandas()
    )

    # Log and register the Spark ML model to Unity Catalog (creates new version automatically-+)
    model_info = mlflow.spark.log_model(
        MLPmodel,
        artifact_path="mlp_model",
        signature=signature,
        input_example=input_example,
        registered_model_name=MLP_MODEL_NAME,
    )

    print(f"Model registered to: {MLP_MODEL_NAME}")
    print(f"Run ID: {run.info.run_id}")
    print(f"Model URI: {model_info.model_uri}")
    print(f"Model version: {model_info.registered_model_version}")
    print(f"AUC: {MLP_auc:.4f}")
    print(f"Recall (label=1): {MLP_recall_1:.4f}")
    print(f"F1 (label=1): {MLP_f1_1:.4f}")

In [0]:
# --- Model Comparison Table ---
comparison_data = [
    ("Random Forest", RF_train_auc, RF_auc, RF_recall_0, RF_recall_1, RF_f1_0, RF_f1_1),
    ("Gradient Boosted Trees", GBT_train_auc, GBT_auc, GBT_recall_0, GBT_recall_1, GBT_f1_0, GBT_f1_1),
    ("MLP Neural Network", MLP_training_auc, MLP_auc, MLP_recall_0, MLP_recall_1, MLP_f1_0, MLP_f1_1),
]

comparison_df = spark.createDataFrame(
    comparison_data,
    ["Model", "Training_AUC", "Test_AUC", "Recall_0", "Recall_1", "F1_0", "F1_1"]
)

print("Model Comparison (15-day fault prediction horizon, 20 lags):")
display(comparison_df)